In [2]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import time

# Pfad zum OSM-Ordner
osm_folder = Path("data/osm")

# Alle PBF-Dateien finden
pbf_files = sorted([f for f in osm_folder.glob("processed_highways_*.pbf")])

print(f"Gefundene PBF-Dateien: {len(pbf_files)}")
for f in pbf_files:
    print(f"  - {f.name}")

# Funktion zum Laden einer einzelnen PBF-Datei
def load_pbf(pbf_file):
    try:
        #gdf = gpd.read_file(pbf_file, layer="lines", columns=["geometry", "name", "highway"])
        gdf = gpd.read_file(pbf_file, layer="lines")
        gdf["region"] = pbf_file.stem.split("_")[-2]  # z.B. "DE-BB" extrahieren
        print(f"  ✓ {pbf_file.name}: {len(gdf)} Zeilen geladen")
        return gdf
    except Exception as e:
        print(f"  ✗ {pbf_file.name}: Fehler: {e}")
        return None

# Paralleles Laden mit ThreadPoolExecutor
start = time.time()
print("\nLade PBF-Dateien parallel...")
with ThreadPoolExecutor(max_workers=4) as executor:
    gdfs = [gdf for gdf in executor.map(load_pbf, pbf_files) if gdf is not None]

# Zu einem GeoDataFrame kombinieren
gdf_all = pd.concat(gdfs, ignore_index=True)
elapsed = time.time() - start

print(f"\n✓ Fertig in {elapsed:.1f}s")
print(f"Gesamt: {len(gdf_all)} Zeilen")
print(f"Spalten: {list(gdf_all.columns)}")
print(f"Regionen: {gdf_all['region'].unique()}")

Gefundene PBF-Dateien: 16
  - processed_highways_DE-BB_latest.pbf
  - processed_highways_DE-BE_latest.pbf
  - processed_highways_DE-BW_latest.pbf
  - processed_highways_DE-BY_latest.pbf
  - processed_highways_DE-HB_latest.pbf
  - processed_highways_DE-HE_latest.pbf
  - processed_highways_DE-HH_latest.pbf
  - processed_highways_DE-MV_latest.pbf
  - processed_highways_DE-NI_latest.pbf
  - processed_highways_DE-NW_latest.pbf
  - processed_highways_DE-RP_latest.pbf
  - processed_highways_DE-SH_latest.pbf
  - processed_highways_DE-SL_latest.pbf
  - processed_highways_DE-SN_latest.pbf
  - processed_highways_DE-ST_latest.pbf
  - processed_highways_DE-TH_latest.pbf

Lade PBF-Dateien parallel...
  ✓ processed_highways_DE-BB_latest.pbf: 1089149 Zeilen geladen
  ✓ processed_highways_DE-BW_latest.pbf: 2317747 Zeilen geladen
  ✓ processed_highways_DE-BY_latest.pbf: 3125917 Zeilen geladen
  ✓ processed_highways_DE-BE_latest.pbf: 458218 Zeilen geladen
  ✓ processed_highways_DE-HB_latest.pbf: 71349 Ze

In [4]:
gdf_all.to_parquet("data/osm/all_highways.parquet", index=False)    

In [5]:
coverage_data = pd.read_csv("data/germany_osm-highways_mp-coverage_latest.csv")

In [6]:
coverage_data.head()

,osm_id,mapillary_coverage
0,1450732492,pano
1,1095301604,pano
2,1095300976,pano
3,1095300975,pano
4,1095202209,pano


In [7]:
gdf_all.osm_id = gdf_all.osm_id.astype(int)
coverage_data.osm_id = coverage_data.osm_id.astype(int)

In [8]:
merged_gdf = gdf_all.merge(coverage_data, on="osm_id", how="left")

In [9]:
merged_gdf

,osm_id,name,highway,waterway,aerialway,barrier,man_made,railway,z_order,other_tags,geometry,region,mapillary_coverage
0,3996955,None,motorway,None,None,None,None,None,9,"""check_date:lit""=>""2021-04-06"",""embankment""=>""...","LINESTRING (13.09264 52.31368, 13.09376 52.315...",DE-BB,regular
1,3996957,None,motorway,None,None,None,None,None,9,"""destination:arrow:lanes""=>""through|through|th...","LINESTRING (13.09522 52.3023, 13.09285 52.30184)",DE-BB,pano
2,4040461,None,motorway,None,None,None,None,None,9,"""int_ref""=>""E 26"",""lanes""=>""2"",""lit""=>""no"",""ma...","LINESTRING (11.92993 53.3017, 11.92475 53.3032...",DE-BB,pano
3,4040465,None,motorway,None,None,None,None,None,9,"""int_ref""=>""E 26"",""lanes""=>""2"",""lit""=>""no"",""ma...","LINESTRING (12.06881 53.28689, 12.06785 53.287...",DE-BB,regular
4,4040467,None,motorway,None,None,None,None,None,9,"""int_ref""=>""E 26"",""lanes""=>""2"",""lit""=>""no"",""ma...","LINESTRING (12.13879 53.26535, 12.13794 53.265...",DE-BB,regular
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16546271,1475095963,None,steps,None,None,None,None,None,0,"""access""=>""private""","LINESTRING (10.73222 50.82282, 10.73231 50.82281)",DE-TH,NaN
16546272,1475095964,None,footway,None,None,None,None,None,0,"""access""=>""private""","LINESTRING (10.73243 50.82301, 10.73267 50.82296)",DE-TH,NaN
16546273,1475095965,None,steps,None,None,None,None,None,0,"""access""=>""private""","LINESTRING (10.73287 50.82273, 10.7328 50.82274)",DE-TH,NaN
16546274,1475095966,None,steps,None,None,None,None,None,0,"""access""=>""private""","LINESTRING (10.73285 50.82268, 10.73278 50.82269)",DE-TH,NaN


In [10]:
merged_gdf_clean=merged_gdf[["osm_id", "highway", "region", "mapillary_coverage", "geometry"]].copy()

In [11]:
merged_gdf_clean.highway.unique()

array(['motorway', 'motorway_link', 'residential', 'primary', 'secondary',
       'tertiary', 'living_street', 'service', 'construction', 'footway',
       'unclassified', 'trunk', 'path', 'secondary_link', 'cycleway',
       'track', 'steps', 'trunk_link', 'pedestrian', 'primary_link',
       'tertiary_link', 'corridor', 'bridleway', 'busway', 'services',
       'proposed', 'raceway', 'platform', 'road', 'rest_area', 'elevator',
       'raised', 'bus_stop', 'street_lamp', 'planned', 'traffic_sign',
       'no', 'crossing', 'demolition', 'emergency_bay', 'escape',
       'disused', 'via_ferrata', 'passing_place', 'traffic_island',
       'traffic_calming', 'ladder', 'yes', 'scramble', 'abandoned:track',
       'demolished:footway', 'emergency', 'hitchhiking', 'access',
       'piste', 'customers', 'patgh', 'st', 'path;bridleway', 'r',
       'emergency_access_point', 'er', 'driveway', 'bus_guideway',
       'death_end', 'residential_link', 'footpath', 'plattform',
       'abandoned', '

In [12]:
merged_gdf_clean_majorRoads=merged_gdf_clean[merged_gdf_clean.highway.isin(['motorway', 'motorway_link',
                                                'trunk', 'trunk_link', 
                                                "primary", "primary_link", 
                                                "secondary", "secondary_link", 
                                                "tertiary", "tertiary_link", 
                                                ])].copy()

In [13]:
merged_gdf_clean_majorRoads

,osm_id,highway,region,mapillary_coverage,geometry
0,3996955,motorway,DE-BB,regular,"LINESTRING (13.09264 52.31368, 13.09376 52.315..."
1,3996957,motorway,DE-BB,pano,"LINESTRING (13.09522 52.3023, 13.09285 52.30184)"
2,4040461,motorway,DE-BB,pano,"LINESTRING (11.92993 53.3017, 11.92475 53.3032..."
3,4040465,motorway,DE-BB,regular,"LINESTRING (12.06881 53.28689, 12.06785 53.287..."
4,4040467,motorway,DE-BB,regular,"LINESTRING (12.13879 53.26535, 12.13794 53.265..."
...,...,...,...,...,...
16546196,1474827547,tertiary,DE-TH,NaN,"LINESTRING (12.57312 50.92311, 12.57347 50.9231)"
16546219,1474895413,tertiary,DE-TH,NaN,"LINESTRING (9.86663 50.68385, 9.86676 50.6839,..."
16546241,1475048515,tertiary,DE-TH,NaN,"LINESTRING (12.46595 50.94996, 12.46601 50.94997)"
16546242,1475048516,tertiary,DE-TH,NaN,"LINESTRING (12.47406 50.97261, 12.47432 50.972..."


In [75]:
kreise=gpd.read_file("https://raw.githubusercontent.com/isellsoap/deutschlandGeoJSON/refs/heads/main/4_kreise/1_sehr_hoch.geo.json")
bland=gpd.read_file("https://raw.githubusercontent.com/isellsoap/deutschlandGeoJSON/refs/heads/main/2_bundeslaender/1_sehr_hoch.geo.json")

In [76]:
kreise

,ID_0,ISO,NAME_0,ID_1,NAME_1,ID_2,NAME_2,ID_3,NAME_3,NL_NAME_3,VARNAME_3,TYPE_3,ENGTYPE_3,geometry
0,86,DEU,Germany,9,Niedersachsen,23,Weser-Ems,244,Oldenburg,None,None,Landkreise,Rural district,"POLYGON ((8.65348 53.11003, 8.66599 53.10659, ..."
1,86,DEU,Germany,9,Niedersachsen,23,Weser-Ems,245,Osnabrück Städte,None,None,Kreisfreie Städte,Urban district,"POLYGON ((7.96379 52.32545, 7.9696 52.32937, 7..."
2,86,DEU,Germany,9,Niedersachsen,23,Weser-Ems,246,Osnabrück,None,None,Landkreise,Rural district,"POLYGON ((8.02655 52.68435, 8.0391 52.67371, 8..."
3,86,DEU,Germany,9,Niedersachsen,23,Weser-Ems,247,Vechta,None,None,Landkreise,Rural district,"POLYGON ((8.46214 52.80015, 8.45627 52.79629, ..."
4,86,DEU,Germany,9,Niedersachsen,23,Weser-Ems,248,Wesermarsch,None,None,Landkreise,Rural district,"MULTIPOLYGON (((8.3075 53.61819, 8.3075 53.617..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
429,86,DEU,Germany,16,Thüringen,40,Thüringen,430,Suhl Städte,None,None,Kreisfreie Städte,Urban district,"POLYGON ((10.73525 50.66061, 10.74113 50.65321..."
430,86,DEU,Germany,16,Thüringen,40,Thüringen,431,Unstrut-Hainich,None,Unstrut-Hainich-Kreis,Landkreise,Rural district,"POLYGON ((10.92664 51.20431, 10.92072 51.19312..."
431,86,DEU,Germany,16,Thüringen,40,Thüringen,432,Wartburgkreis,None,None,Landkreise,Rural district,"POLYGON ((10.61533 51.03504, 10.60371 51.02757..."
432,86,DEU,Germany,16,Thüringen,40,Thüringen,433,Weimar Städte,None,None,Kreisfreie Städte,Urban district,"POLYGON ((11.25191 51.03471, 11.25189 51.03099..."


In [77]:
kreise.rename(columns={"NAME_1":"Bundesland","NAME_3": "Landkreis"}, inplace=True)
bland.rename(columns={"name":"Bundesland"}, inplace=True)


In [78]:
bland

,id,Bundesland,type,geometry
0,DE-BW,Baden-Württemberg,State,"MULTIPOLYGON (((8.70837 47.71556, 8.70918 47.7..."
1,DE-BY,Bayern,State,"POLYGON ((10.13386 50.55, 10.1398 50.54252, 10..."
2,DE-BE,Berlin,State,"POLYGON ((13.16181 52.59442, 13.174 52.59425, ..."
3,DE-BB,Brandenburg,State,"POLYGON ((13.87951 53.50107, 13.87927 53.49908..."
4,DE-HB,Bremen,State,"POLYGON ((8.98545 53.12822, 8.97316 53.12799, ..."
5,DE-HH,Hamburg,State,"POLYGON ((10.07162 53.71823, 10.0715 53.72192,..."
6,DE-HE,Hessen,State,"POLYGON ((9.49877 51.63152, 9.50474 51.62795, ..."
7,DE-MV,Mecklenburg-Vorpommern,State,"MULTIPOLYGON (((14.26472 53.71069, 14.26472 53..."
8,DE-NI,Niedersachsen,State,"MULTIPOLYGON (((6.86528 53.59597, 6.86528 53.5..."
9,DE-NW,Nordrhein-Westfalen,State,"POLYGON ((8.66628 52.52528, 8.67277 52.51795, ..."


In [79]:
target_crs = 25832
lines = merged_gdf_clean_majorRoads.to_crs(target_crs)
kreise  = kreise.to_crs(target_crs)  # kreise: Polygon-GDF mit Spalte "Landkreis"


In [80]:
pts = lines[["osm_id", "geometry"]].copy()
pts["geometry"] = pts.geometry.representative_point()


In [81]:
joined = pts.sjoin(
    kreise[["Landkreis","Bundesland", "geometry"]],
    how="left",
    predicate="within",
)

In [82]:
lines = lines.merge(
    joined[["osm_id", "Landkreis","Bundesland"]],
    on="osm_id",
    how="left",
)


In [83]:
lines

,osm_id,highway,region,mapillary_coverage,geometry,Landkreis,Bundesland
0,3996955,motorway,DE-BB,regular,"LINESTRING (778933.815 5803816.85, 778999.282 ...",Potsdam-Mittelmark,Brandenburg
1,3996957,motorway,DE-BB,pano,"LINESTRING (779181.582 5802561.592, 779023.187...",Potsdam-Mittelmark,Brandenburg
2,4040461,motorway,DE-BB,pano,"LINESTRING (695230.778 5909836.658, 694878.545...",Parchim,Mecklenburg-Vorpommern
3,4040461,motorway,DE-BB,pano,"LINESTRING (695230.778 5909836.658, 694878.545...",Parchim,Mecklenburg-Vorpommern
4,4040465,motorway,DE-BB,regular,"LINESTRING (704553.042 5908579.118, 704487.691...",Prignitz,Brandenburg
...,...,...,...,...,...,...,...
1606488,1474895413,tertiary,DE-TH,NaN,"LINESTRING (561223.013 5615027.469, 561232.308...",Fulda,Hessen
1606489,1474895413,tertiary,DE-TH,NaN,"LINESTRING (561223.013 5615027.469, 561232.308...",Fulda,Hessen
1606490,1475048515,tertiary,DE-TH,NaN,"LINESTRING (743434.92 5649981.946, 743439.38 5...",Altenburger Land,Thüringen
1606491,1475048516,tertiary,DE-TH,NaN,"LINESTRING (743885.838 5652526.263, 743904.698...",Altenburger Land,Thüringen


In [84]:
lines["length_m"] = lines.geometry.length

In [85]:
lines

,osm_id,highway,region,mapillary_coverage,geometry,Landkreis,Bundesland,length_m
0,3996955,motorway,DE-BB,regular,"LINESTRING (778933.815 5803816.85, 778999.282 ...",Potsdam-Mittelmark,Brandenburg,489.512134
1,3996957,motorway,DE-BB,pano,"LINESTRING (779181.582 5802561.592, 779023.187...",Potsdam-Mittelmark,Brandenburg,169.464392
2,4040461,motorway,DE-BB,pano,"LINESTRING (695230.778 5909836.658, 694878.545...",Parchim,Mecklenburg-Vorpommern,2089.505867
3,4040461,motorway,DE-BB,pano,"LINESTRING (695230.778 5909836.658, 694878.545...",Parchim,Mecklenburg-Vorpommern,2089.505867
4,4040465,motorway,DE-BB,regular,"LINESTRING (704553.042 5908579.118, 704487.691...",Prignitz,Brandenburg,5345.608668
...,...,...,...,...,...,...,...,...
1606488,1474895413,tertiary,DE-TH,NaN,"LINESTRING (561223.013 5615027.469, 561232.308...",Fulda,Hessen,614.275539
1606489,1474895413,tertiary,DE-TH,NaN,"LINESTRING (561223.013 5615027.469, 561232.308...",Fulda,Hessen,614.275539
1606490,1475048515,tertiary,DE-TH,NaN,"LINESTRING (743434.92 5649981.946, 743439.38 5...",Altenburger Land,Thüringen,4.590947
1606491,1475048516,tertiary,DE-TH,NaN,"LINESTRING (743885.838 5652526.263, 743904.698...",Altenburger Land,Thüringen,151.444109


In [89]:
import pandas as pd

df = lines.copy()

# NaN explizit
df["mapillary_coverage"] = df["mapillary_coverage"].fillna("NaN")

# _link entfernen → motorway_link → motorway
df["highway"] = df["highway"].str.replace("_link$", "", regex=True)


In [90]:
df

,osm_id,highway,region,mapillary_coverage,geometry,Landkreis,Bundesland,length_m
0,3996955,motorway,DE-BB,regular,"LINESTRING (778933.815 5803816.85, 778999.282 ...",Potsdam-Mittelmark,Brandenburg,489.512134
1,3996957,motorway,DE-BB,pano,"LINESTRING (779181.582 5802561.592, 779023.187...",Potsdam-Mittelmark,Brandenburg,169.464392
2,4040461,motorway,DE-BB,pano,"LINESTRING (695230.778 5909836.658, 694878.545...",Parchim,Mecklenburg-Vorpommern,2089.505867
3,4040461,motorway,DE-BB,pano,"LINESTRING (695230.778 5909836.658, 694878.545...",Parchim,Mecklenburg-Vorpommern,2089.505867
4,4040465,motorway,DE-BB,regular,"LINESTRING (704553.042 5908579.118, 704487.691...",Prignitz,Brandenburg,5345.608668
...,...,...,...,...,...,...,...,...
1606488,1474895413,tertiary,DE-TH,NaN,"LINESTRING (561223.013 5615027.469, 561232.308...",Fulda,Hessen,614.275539
1606489,1474895413,tertiary,DE-TH,NaN,"LINESTRING (561223.013 5615027.469, 561232.308...",Fulda,Hessen,614.275539
1606490,1475048515,tertiary,DE-TH,NaN,"LINESTRING (743434.92 5649981.946, 743439.38 5...",Altenburger Land,Thüringen,4.590947
1606491,1475048516,tertiary,DE-TH,NaN,"LINESTRING (743885.838 5652526.263, 743904.698...",Altenburger Land,Thüringen,151.444109


In [106]:
r_einheit="Landkreis" # "Bundesland"
#r_einheit="Bundesland" # "Bundesland"

In [107]:
agg = (
    df.groupby([r_einheit, "highway", "mapillary_coverage"], observed=True)["length_m"]
      .sum()
      .reset_index()
)

agg["total_length"] = (
    agg.groupby([r_einheit, "highway"])["length_m"]
       .transform("sum")
)

agg["share"] = agg["length_m"] / agg["total_length"]


In [108]:
agg

,Landkreis,highway,mapillary_coverage,length_m,total_length,share
0,Aachen,motorway,NaN,13546.467546,54961.542751,0.246472
1,Aachen,motorway,regular,41415.075205,54961.542751,0.753528
2,Aachen,primary,NaN,53336.775187,128756.993552,0.414244
3,Aachen,primary,pano,396.520423,128756.993552,0.003080
4,Aachen,primary,regular,75023.697942,128756.993552,0.582677
...,...,...,...,...,...,...
4877,Zwickauer Land,tertiary,NaN,232663.045325,299372.421016,0.777169
4878,Zwickauer Land,tertiary,pano,3617.630172,299372.421016,0.012084
4879,Zwickauer Land,tertiary,regular,63091.745519,299372.421016,0.210747
4880,Zwickauer Land,trunk,NaN,11281.035779,44276.830095,0.254784


In [109]:
agg_all = (
    df.groupby([r_einheit, "mapillary_coverage"], observed=True)["length_m"]
      .sum()
      .reset_index()
)

agg_all["total_length"] = (
    agg_all.groupby(r_einheit)["length_m"]
           .transform("sum")
)

agg_all["share"] = agg_all["length_m"] / agg_all["total_length"]

# highway-Spalte ergänzen
agg_all["highway"] = "all"


In [110]:
agg_all

,Landkreis,mapillary_coverage,length_m,total_length,share,highway
0,Aachen,NaN,371716.783720,594555.814283,0.625201,all
1,Aachen,pano,803.546753,594555.814283,0.001352,all
2,Aachen,regular,222035.483810,594555.814283,0.373448,all
3,Aachen Städte,NaN,131933.787180,319516.989224,0.412916,all
4,Aachen Städte,pano,378.333904,319516.989224,0.001184,all
...,...,...,...,...,...,...
1163,Zwickau Städte,NaN,35235.395387,90313.936326,0.390144,all
1164,Zwickau Städte,regular,55078.540939,90313.936326,0.609856,all
1165,Zwickauer Land,NaN,323568.300249,660691.669823,0.489742,all
1166,Zwickauer Land,pano,33471.725414,660691.669823,0.050662,all


In [111]:
agg_combined = pd.concat([agg, agg_all], ignore_index=True)

In [112]:
result = (
    agg_combined
      .pivot(
          index=[r_einheit, "highway"],
          columns="mapillary_coverage",
          values="share"
      )
      .fillna(0)
      .reset_index()
)


In [113]:
result

mapillary_coverage,Landkreis,highway,NaN,pano,regular
0,Aachen,all,0.625201,0.001352,0.373448
1,Aachen,motorway,0.246472,0.000000,0.753528
2,Aachen,primary,0.414244,0.003080,0.582677
3,Aachen,secondary,0.728315,0.000449,0.271236
4,Aachen,tertiary,0.768131,0.001842,0.230027
...,...,...,...,...,...
2421,Zwickauer Land,motorway,0.013813,0.223063,0.763123
2422,Zwickauer Land,primary,0.224476,0.000000,0.775524
2423,Zwickauer Land,secondary,0.358785,0.059368,0.581847
2424,Zwickauer Land,tertiary,0.777169,0.012084,0.210747


In [114]:
order = ["motorway", "trunk", "primary", "secondary", "tertiary", "all"]
result["highway"] = pd.Categorical(result["highway"], order, ordered=True)
result = result.sort_values([r_einheit, "highway"])

In [115]:
result

mapillary_coverage,Landkreis,highway,NaN,pano,regular
1,Aachen,motorway,0.246472,0.000000,0.753528
5,Aachen,trunk,0.000000,0.000000,1.000000
2,Aachen,primary,0.414244,0.003080,0.582677
3,Aachen,secondary,0.728315,0.000449,0.271236
4,Aachen,tertiary,0.768131,0.001842,0.230027
...,...,...,...,...,...
2425,Zwickauer Land,trunk,0.254784,0.000000,0.745216
2422,Zwickauer Land,primary,0.224476,0.000000,0.775524
2423,Zwickauer Land,secondary,0.358785,0.059368,0.581847
2424,Zwickauer Land,tertiary,0.777169,0.012084,0.210747


In [116]:
import pandas as pd

r = result.copy().rename(columns={"NaN": "no_cover"})

wide_long = r.melt(
    id_vars=[r_einheit, "highway"],
    value_vars=["no_cover", "pano", "regular"],
    var_name="coverage",
    value_name="share",
)

# FIX: Categorical -> string
wide_long["col"] = (
    wide_long["highway"].astype("string") + "_" + wide_long["coverage"].astype("string")
)

wide = (
    wide_long.pivot(index=r_einheit, columns="col", values="share")
             .fillna(0)
             .reset_index()
)

wide.head()


col,Landkreis,all_no_cover,all_pano,all_regular,motorway_no_cover,motorway_pano,motorway_regular,primary_no_cover,primary_pano,primary_regular,secondary_no_cover,secondary_pano,secondary_regular,tertiary_no_cover,tertiary_pano,tertiary_regular,trunk_no_cover,trunk_pano,trunk_regular
0,Aachen,0.625201,0.001352,0.373448,0.246472,0.000000,0.753528,0.414244,0.003080,0.582677,0.728315,0.000449,0.271236,0.768131,0.001842,0.230027,0.000000,0.0,1.000000
1,Aachen Städte,0.412916,0.001184,0.585900,0.225764,0.000000,0.774236,0.229755,0.003384,0.766861,0.462175,0.000628,0.537197,0.651702,0.000820,0.347478,0.461447,0.0,0.538553
2,Ahrweiler,0.434921,0.000000,0.565079,0.106252,0.000000,0.893748,0.490484,0.000000,0.509516,0.337266,0.000000,0.662734,0.572466,0.000000,0.427534,0.609346,0.0,0.390654
3,Aichach-Friedberg,0.709539,0.042604,0.247856,0.054665,0.476093,0.469242,0.684622,0.000000,0.315378,0.534417,0.000611,0.464972,0.938865,0.000000,0.061135,0.970614,0.0,0.029386
4,Alb-Donau,0.475545,0.054967,0.469488,0.035381,0.000000,0.964619,0.346687,0.103968,0.549345,0.427233,0.057025,0.515742,0.639994,0.051845,0.308161,0.098064,0.0,0.901936


In [117]:
'''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''

SyntaxError: unterminated string literal (detected at line 1) (3726331259.py, line 1)

In [118]:
kreise_wide = kreise[["Landkreis","geometry"]].merge(wide, on="Landkreis", how="left").fillna(0)

In [119]:
kreise_wide

,Landkreis,geometry,all_no_cover,all_pano,all_regular,motorway_no_cover,motorway_pano,motorway_regular,primary_no_cover,primary_pano,primary_regular,secondary_no_cover,secondary_pano,secondary_regular,tertiary_no_cover,tertiary_pano,tertiary_regular,trunk_no_cover,trunk_pano,trunk_regular
0,Oldenburg,"POLYGON ((476804.089 5884566.798, 477639.735 5...",0.373649,0.038282,0.588068,0.075276,0.168264,0.756460,0.025378,0.001409,0.973213,0.431665,0.000000,0.568335,0.532729,0.000091,0.467180,0.057775,0.000000,0.942225
1,Osnabrück Städte,"POLYGON ((429381.607 5797742.247, 429783.773 5...",0.499394,0.000000,0.500606,0.228988,0.000000,0.771012,0.129902,0.000000,0.870098,0.611316,0.000000,0.388684,0.732861,0.000000,0.267139,0.690336,0.000000,0.309664
2,Osnabrück,"POLYGON ((434196.757 5837602.494, 435029.329 5...",0.690068,0.000000,0.309932,0.089229,0.000000,0.910771,0.668520,0.000000,0.331480,0.721967,0.000000,0.278033,0.813641,0.000000,0.186359,0.496377,0.000000,0.503623
3,Vechta,"POLYGON ((463737.65 5850175.018, 463338.599 58...",0.592587,0.000000,0.407413,0.043602,0.000000,0.956398,0.113981,0.000000,0.886019,0.604093,0.000000,0.395907,0.787355,0.000000,0.212645,0.000000,0.000000,0.000000
4,Wesermarsch,"MULTIPOLYGON (((454192.912 5941266.035, 454192...",0.509700,0.029452,0.460849,0.000000,0.000000,0.000000,0.449808,0.069235,0.480957,0.458416,0.009218,0.532366,0.574853,0.032696,0.392450,0.335579,0.000000,0.664421
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
429,Suhl Städte,"POLYGON ((622644.171 5613521.12, 623079.103 56...",0.341627,0.000000,0.658373,0.372784,0.000000,0.627216,0.000000,0.000000,0.000000,0.326360,0.000000,0.673640,0.223676,0.000000,0.776324,0.000000,0.000000,0.000000
430,Unstrut-Hainich,"POLYGON ((634591.428 5674309.451, 634210.398 5...",0.667978,0.007769,0.324254,0.000000,0.000000,0.000000,0.349175,0.002744,0.648081,0.684042,0.013218,0.302740,0.909731,0.000522,0.089748,0.552164,0.000000,0.447836
431,Wartburgkreis,"POLYGON ((613258.516 5654963.187, 612461.878 5...",0.738644,0.022306,0.239050,0.085985,0.229451,0.684564,0.681054,0.000000,0.318946,0.828688,0.000398,0.170914,0.910720,0.000000,0.089280,0.396410,0.000000,0.603590
432,Weimar Städte,"POLYGON ((657889.108 5656097.825, 657900.481 5...",0.527515,0.051301,0.421184,0.057627,0.000000,0.942373,0.651733,0.000000,0.348267,0.075346,0.009483,0.915171,0.660294,0.093095,0.246610,0.000000,0.000000,0.000000


In [120]:

kreise_wide_4326=kreise_wide.to_crs(4326)
kreise_wide_4326.to_file("data/kreise_wide.fgb", driver="FlatGeobuf")

In [130]:
def fgb_to_pmtiles_layer(polys_fgb, output_pmtiles, layer_name):
    import subprocess
    from pathlib import Path

    subprocess.run([
        "tippecanoe", "-o", str(Path(output_pmtiles).resolve()),
        f"--layer={layer_name}",
        "--minimum-zoom=5", "--maximum-zoom=7",
        "--force",
        "--no-feature-limit", "--no-tile-size-limit", 
        "--drop-densest-as-needed",  # Nur Punkte entfernen, keine Vereinfachung
        str(Path(polys_fgb).resolve())
    ], check=True)

    print("✅ Combined PMTiles created")

# Tippecanoe macht automatisch zoom-basierte Vereinfachung ohne Topologie-Probleme
fgb_to_pmtiles_layer(
    polys_fgb="data/kreise_wide.fgb",
    output_pmtiles="data/kreise_wide.pmtiles",
    layer_name="default"
)

detected indexed FlatGeobuf: assigning feature IDs by sequence
434 features, 596300 bytes of geometry and attributes, 132366 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes


✅ Combined PMTiles created


detected indexed FlatGeobuf: assigning feature IDs by sequence
434 features, 754992 bytes of geometry and attributes, 138417 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes


✅ Combined PMTiles created


  99.9%  10/540/326  


In [ ]:
bland_wide = bland[["Bundesland","geometry"]].merge(wide, on="Bundesland", how="left").fillna(0)

bland_wide_4326=bland_wide.to_crs(4326)

bland_wide_4326.to_file("data/bland_wide.fgb", driver="FlatGeobuf")

# Tippecanoe vereinfacht automatisch zoom-basiert
fgb_to_pmtiles_layer(
    polys_fgb="data/bland_wide.fgb",
    output_pmtiles="data/bland_wide.pmtiles",
    layer_name="default"
)

detected indexed FlatGeobuf: assigning feature IDs by sequence
16 features, 305645 bytes of geometry and attributes, 8418 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes
  99.9%  8/137/81  


✅ Combined PMTiles created
